In [30]:
# @title Imports

from IPython import display

from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from random import randint, choice
import sentence_transformers
from threading import Lock
from typing import Callable

from concordia.associative_memory.associative_memory import AssociativeMemory
from concordia.associative_memory.blank_memories import MemoryFactory
from concordia.associative_memory.formative_memories import AgentConfig
from concordia.associative_memory.formative_memories import FormativeMemoryFactory
from concordia.associative_memory.importance_function import ConstantImportanceModel
from concordia.agents.basic_agent import BasicAgent
from concordia.clocks.game_clock import FixedIntervalClock
from concordia import components as components
from concordia.components import agent as agent_components
from concordia.components import game_master as gm_components
from concordia.document.interactive_document import InteractiveDocument
from concordia.environment.game_master import GameMaster
from concordia.language_model.gpt_model import GptLanguageModel
from concordia.language_model.language_model import LanguageModel
from concordia.typing.component import Component


In [31]:
# @title Setup Encoder and Model
OPENAI_KEY = ""

embedder_model = sentence_transformers\
    .SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
embedder = lambda x: embedder_model.encode(x, show_progress_bar=False)

model = GptLanguageModel(
    api_key = OPENAI_KEY,
    model_name = "gpt-3.5-turbo-0125"
)

/Users/lancylot2004/miniconda3/envs/Concordia/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [32]:
# @title Shared Knowledge of Coin Game

COIN_GRID_WIDTH, COIN_GRID_HEIGHT = 3, 3

coin_rules = [
    'There are two players in the game - Red and Blue.',
    f'The game occurs in a {COIN_GRID_WIDTH}x{COIN_GRID_HEIGHT} grid.',
    'The game occurs in rounds.',
    'In each round, one coin is placed randomly in the grid.',
    'The coin is either red or blue.',
    'The players initially start at the top left and bottom right of the grid.',
    'The players move simultaneously and can overlap each other.',
    'The players can move up, down, left, or right, within the grid.',
    'The round ends when one or both of the player reaches the coin.',
    'Any player which reaches the coin gets one point.',
    'If the coin they picked up is not their color, the other player loses two points.',
    'The next round starts with the players in their current position and a new coin somewhere else.'
    'The goal is to maximise the points of BOTH players.'
]

coin_context = model.sample_text(
    'Summarise the following rules in a concise and insightful manner:\n'
    + '\n'.join(coin_rules)
)
print(f"Summary: {coin_context}")

Summary: In a game with Red and Blue players on a grid, coins are placed randomly each round. Players move simultaneously to reach the coin for points, but lose points if they pick up the wrong colored coin. The strategy is to maximize points for both players while navigating the grid.


In [33]:
# @title Setup Clock

INIT_TIME = datetime(year = 2024, month = 7, day = 3, hour = 10)

clock = FixedIntervalClock(
    start = INIT_TIME,
    step_size = timedelta(minutes = 1)
)

In [34]:
# @title Setup Memory

player_importance = ConstantImportanceModel()
gm_importance = ConstantImportanceModel()

blank_factory = MemoryFactory(
    model = model,
    embedder = embedder,
    importance = player_importance.importance,
    clock_now = clock.now,
)

formative_factory = FormativeMemoryFactory(
    model = model,
    shared_memories = coin_rules,
    blank_memory_factory_call = blank_factory.make_blank_memory
)

In [36]:
# @title Coin Component

"""Tracks the current state of the coin game."""
class CoinStatus(Component):
    def __init__(
        self,
        clock_now: Callable[[], datetime],
        model: LanguageModel,
        red_memory: AssociativeMemory,
        blue_memory: AssociativeMemory,
        verbose: bool = True,
    ) -> None:
        super().__init__()
        self._lock = Lock()
        self._clock_call = clock_now
        self._model = model
        self._grid = [[None for _ in range(COIN_GRID_WIDTH)] for _ in range(COIN_GRID_HEIGHT)]
        self._red_points, self._blue_points = 0, 0
        self._red_memory, self._blue_memory = red_memory, blue_memory
        self._red_position = (0, 0)
        self._blue_position = (COIN_GRID_WIDTH - 1, COIN_GRID_HEIGHT - 1)
        self._coin_position = None
        self._coin_color = None
        self._verbose = verbose
        self._history = []

    # Override
    def name(self) -> str:
        return "Status of Coin Game"

    # Override
    def state(self) -> str:
        with self._lock:
            return f"Red: {self._red_points}, Blue: {self._blue_points}\n"\
                   + '\n'.join([''.join([str(cell) for cell in row]) for row in self._grid])

    # Override
    def partial_state(self, player_name: str) -> str | None:
        return self.state()

    # Override
    def observe(self, observation: str) -> None:
        with self._lock:
            # Handle observation input to update the game state, e.g., player positions, coin state, etc.
            pass

    # Override
    def update(self) -> None:
        with self._lock:
            self._update_player_state("Red", self._red_memory)
            self._update_player_state("Blue", self._blue_memory)

    def _update_player_state(self, player_name: str, memory: AssociativeMemory) -> None:
        action = memory.retrieve_recent()
        prompt = InteractiveDocument(self._model)
        prompt.statement(f"Action: {action}\n")
        direction = prompt.open_question(
            f'Given the above action carried out by {player_name}, in',
            f'which direction did {player_name} move? Answer with "up"',
            f'"down", "left", or "right".',
        )

        self.move_player(player_name, direction)
        if self._verbose:
            print(prompt.view().text())

        self._history.append({
            'date': self._clock_call(),
            'state': self.state(),
            'player': player_name,
            'status': f'Moved {direction}',
        })

    # Override
    def update_before_event(self, cause_statement: str) -> None:
        with self._lock:
            pass

    # Override
    def update_after_event(self, event_statement: str) -> None:
        with self._lock:
            pass

    # Override
    def terminate_episode(self) -> bool:
        return self._blue_position == self._coin_position \
          or self._red_position == self._coin_position

    def get_last_log(self):
        with self._lock:
            if self._history:
                return self._history[-1].copy()

    def place_coin(self, position: tuple, color: str) -> None:
        with self._lock:
            self._coin_position = position
            self._coin_color = color
            self._grid[position[1]][position[0]] = color

    def move_player(self, player_name: str, direction: str) -> None:
        with self._lock:
            if player_name == "Red":
                current_position = self._red_position
            elif player_name == "Blue":
                current_position = self._blue_position
            else:
                return

            new_position = self._calculate_new_position(current_position, direction)
            if self._is_position_valid(new_position):
                if player_name == "Red":
                    self._red_position = new_position
                elif player_name == "Blue":
                    self._blue_position = new_position

            self._check_coin_reached()

    def _calculate_new_position(self, position: tuple, direction: str) -> tuple:
        x, y = position
        match direction.lower():
            case "up": y = max(0, y - 1)
            case "down": y = min(COIN_GRID_HEIGHT - 1, y + 1)
            case "left": x = max(0, x - 1)
            case "right": x = min(COIN_GRID_WIDTH - 1, x + 1)
            case _: raise ValueError(f"Invalid direction: {direction}")
        return (x, y)

    def _is_position_valid(self, position: tuple) -> bool:
        x, y = position
        return 0 <= x < COIN_GRID_WIDTH and 0 <= y < COIN_GRID_HEIGHT

    def _check_coin_reached(self) -> None:
        if self._red_position == self._coin_position:
            self._handle_coin_pickup("Red")
        if self._blue_position == self._coin_position:
            self._handle_coin_pickup("Blue")

    def _handle_coin_pickup(self, player_name: str) -> None:
        if player_name == "Red":
            if self._coin_color == "red":
                self._red_points += 1
            else:
                self._red_points += 1
                self._blue_points -= 2
        elif player_name == "Blue":
            if self._coin_color == "blue":
                self._blue_points += 1
            else:
                self._blue_points += 1
                self._red_points -= 2

        # Clear the coin from the grid
        self._grid[self._coin_position[1]][self._coin_position[0]] = None
        self._coin_position = None
        self._coin_color = None
        self._place_new_coin()

    def _place_new_coin(self) -> None:
        x, y = randint(0, COIN_GRID_WIDTH - 1), randint(0, COIN_GRID_HEIGHT - 1)
        color = choice(["red", "blue"])
        self.place_coin((x, y), color)


In [37]:
# @title Agent Building

"""Builds an agent to play the Coin game with the given configuration.

Args:
    config (AgentConfig): The configuration of the agent.

Returns:
    (BasicAgent, AssociativeMemory): The agent and its memory object.
"""
def build_agent(config: AgentConfig) -> tuple[BasicAgent, AssociativeMemory]:
    memory = formative_factory.make_memories(config)

    timeComp = agent_components.report_function.ReportFunction(
        name = "Current Time",
        function = clock.current_time_interval_str,
    )

    idComp = agent_components.identity.SimIdentity(model, memory, config.name)
    planComp = agent_components.plan.SimPlan(
        model = model, memory = memory, agent_name = config.name,
        clock_now = clock.now, components = [idComp], verbose = False,
        goal = agent_components.constant.ConstantComponent(state = config.goal),
    )

    observeComp = agent_components.observation.Observation(
        agent_name = config.name, memory = memory, clock_now = clock.now,
        timeframe = clock.get_step_size(), component_name = "Observation",
    )
    summaryComp = agent_components.observation.ObservationSummary(
        agent_name = config.name, model = model, clock_now = clock.now,
        memory = memory, components = [idComp], component_name = "Summary",
        timeframe_delta_from = timedelta(minutes = 10),
        timeframe_delta_until = timedelta(minutes = 0),
    )

    agent = BasicAgent(
        agent_name = config.name, model = model, clock = clock, verbose = True,
        components = [idComp, planComp, timeComp, observeComp, summaryComp]
    )

    return agent, memory

# TODO: Add "interesting" traits to experiment.
player_configs =  map(
    lambda name: AgentConfig(
        name = name,
        goal = "Maximise every player's points.",
        context = coin_context
    ),
    ["Red", "Blue"]
)

players = {}
with ThreadPoolExecutor(max_workers = 2) as executor:
    for agent, memory in executor.map(build_agent, player_configs):
        players[agent.name] = {'player': agent, 'memory': memory}


In [41]:
# @title Game Master Building

gm_memory = AssociativeMemory(
    sentence_embedder = embedder,
    importance = gm_importance.importance,
    clock = clock.now,
)

rulesComp = agent_components.constant.ConstantComponent(
    state = '\n'.join(coin_rules), name = "Coin Game Rules"
)

coinComp = CoinStatus(
    clock_now = clock.now,
    model = model,
    red_memory = players["Red"]['memory'],
    blue_memory = players["Blue"]['memory'],
    verbose = True,
)

env = GameMaster(
    model = model,
    memory = gm_memory,
    clock = clock,
    components = [rulesComp, coinComp],
    players = list(map(lambda player: player['player'], players.values())),
    verbose = True,
)

In [42]:
for _ in range(5):
  env.step()

# @title Summarize the entire story
all_gm_memories = env._memory.retrieve_recent(k=10000, add_time=True)

detailed_story = '\n'.join(all_gm_memories)
print('len(detailed_story): ', len(detailed_story))
# print(detailed_story)

episode_summary = model.sample_text(
    f'Sequence of events:\n{detailed_story}'+
    '\nNarratively summarize the above temporally ordered ' +
    'sequence of events. Write it as a news report. Summary:\n',
     max_tokens=3500, terminators=())
print(episode_summary)


Red context of action:
Red's Current Time:
 03 Jul 2024 [10:00:00 - 10:01:00]

Red's Observation:
[03 Jul 2024 10:00:00] [observation] Red: 0, Blue: 0
[03 Jul 2024 10:00:00] [observation] NoneNoneNone




GM context of action and chain of thought:
Instructions: This is a social science experiment. It is structured as a tabletop roleplaying game (like dungeons and dragons). You are the game master. You will describe the current situation to the participants in the experiment and then on the basis of what you tell them they will suggest actions for the character they control. Aside from you, each other participant controls just one character. You are the game master so you may control any non-player character. You will track the state of the world and keep it consistent as time passes in the simulation and the participants take actions and change things in their world. Remember that this is a serious social science experiment. It is not just a game. It need not be fun for the participan